# HydroSense AI - Exploratory Data Analysis

This notebook reports per-cycle summaries, raw signal patterns, correlations, target relationships, and outlier counts. It does not modify data or train models.

## 1. Setup

In [1]:
from pathlib import Path
import sys
import gc

import pandas as pd
import matplotlib.pyplot as plt

# Find the project root whether the notebook runs from /notebooks or from the root
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

# Make the src package importable
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Python used by this kernel:", sys.executable)
print("Project root:", PROJECT_ROOT)
print("pandas version:", pd.__version__)

Python used by this kernel: c:\Users\o3995\anaconda3\python.exe
Project root: C:\Users\o3995\Downloads\ML_SUT_Projects\Individual_Lab_ML_SUT\Hydraulic_Systems\System
pandas version: 2.3.3


In [2]:
from src import config
from src.data_loading import load_profile, load_sensor, validate_dataset
from src.eda import (
    compute_cycle_summary,
    count_outliers_iqr,
    check_label_combinations,
    row_order_check,
    sensor_quality_checks,
)
from src.utils import set_plot_style, save_figure

set_plot_style()

## 2. Descriptive statistics of per-cycle means

In [3]:
profile = load_profile()
summary = compute_cycle_summary()
means = summary[[f"{sensor}_mean" for sensor in config.SENSORS]].copy()
means.columns = list(config.SENSORS)
print("EDA summary shape:", summary.shape)

descriptive = means.describe().T
descriptive.to_csv(config.RESULTS_DIR / "descriptive_statistics.csv")
display(descriptive)

EDA summary shape: (2205, 68)


,count,mean,std,min,25%,50%,75%,max
PS1,2205.0,160.485315,4.699424,155.391547,158.100195,158.960895,161.000736,180.922709
PS2,2205.0,109.379906,4.986585,104.406307,106.962383,107.730169,109.421612,131.589089
PS3,2205.0,1.753227,0.251902,0.840253,1.729733,1.779630,1.932047,2.023398
PS4,2205.0,2.600266,4.279355,0.000000,0.000000,0.000000,3.503266,10.207068
PS5,2205.0,9.163320,0.576296,8.365800,8.547239,9.115781,9.844351,9.978510
PS6,2205.0,9.079363,0.549589,8.321527,8.487167,9.031516,9.729276,9.856591
EPS1,2205.0,2495.509224,73.836683,2361.747289,2442.933474,2480.926667,2548.211492,2740.641032
FS1,2205.0,6.198549,1.032883,2.018572,6.391670,6.576673,6.657508,6.722707
FS2,2205.0,9.649453,0.449246,8.857513,9.203397,9.692270,10.155008,10.403098
TS1,2205.0,45.424567,7.991933,35.313784,36.237150,44.836650,54.104316,57.899283


## 3. Distribution plots

In [4]:
import seaborn as sns
groups = {
    "pressure": [f"PS{i}" for i in range(1, 7)],
    "flow_power": ["EPS1", "FS1", "FS2"],
    "low_rate": ["TS1", "TS2", "TS3", "TS4", "VS1", "CE", "CP", "SE"],
}
for group_name, columns in groups.items():
    fig, axes = plt.subplots((len(columns) + 2) // 3, 3, figsize=(13, 3 * ((len(columns) + 2) // 3)))
    axes = axes.ravel()
    for ax, sensor in zip(axes, columns):
        sns.histplot(means[sensor], kde=True, ax=ax)
        ax.set_title(sensor)
    for ax in axes[len(columns):]:
        ax.set_visible(False)
    save_figure(fig, f"02_sensor_mean_distributions_{group_name}")
    plt.close(fig)
print("Saved 02_sensor_mean_distributions_pressure.png, flow_power.png, and low_rate.png")

Saved 02_sensor_mean_distributions_pressure.png, flow_power.png, and low_rate.png


### Observation
Per-cycle means range from 0 to 180.9 bar for PS1 and reach 2740.6 W for EPS1. Temperature, cooling, pressure, and efficiency groups show clear class-related spread; distributions will be interpreted with target groups below.

## 4. Controlled raw signal examples
For each component, example cycles prefer the healthy/common setting for the other three condition targets.

In [5]:
healthy = {"cooler": 100, "valve": 100, "pump": 0, "accumulator": 130}
examples = {}
for target in config.TARGET_COLUMNS[:4]:
    target_rows = {}
    for value in sorted(profile[target].unique()):
        mask = profile[target].eq(value)
        for other, setting in healthy.items():
            if other != target:
                mask &= profile[other].eq(setting)
        candidates = profile.index[mask]
        if len(candidates) == 0:
            candidates = profile.index[profile[target].eq(value)]
        target_rows[int(value)] = int(candidates[0])
    examples[target] = target_rows
pd.DataFrame(examples).to_csv(config.RESULTS_DIR / "example_cycle_rows.csv", index=False)
print(pd.DataFrame(examples))

     cooler   valve    pump  accumulator
3       0.0     NaN     NaN          NaN
20    932.0     NaN     NaN          NaN
100  1664.0  1664.0     NaN       1930.0
73      NaN  1757.0     NaN          NaN
80      NaN  1767.0     NaN          NaN
90      NaN  1777.0     NaN       1464.0
0       NaN     NaN  1664.0          NaN
1       NaN     NaN  1715.0          NaN
2       NaN     NaN  1674.0          NaN
115     NaN     NaN     NaN       1797.0
130     NaN     NaN     NaN       1664.0


In [6]:
def plot_raw(sensor, target, filename):
    values = load_sensor(sensor)
    meta = config.SENSORS[sensor]
    time = pd.Series(range(values.shape[1])) / meta["sampling_hz"]
    fig, ax = plt.subplots(figsize=(11, 4))
    for value, row in examples[target].items():
        ax.plot(time, values[row], label=f"{target}={value}, row={row}")
    ax.set(xlabel="seconds", ylabel=meta["unit"], title=f"{sensor} for {target} classes")
    ax.legend(ncol=2)
    save_figure(fig, filename)
    plt.close(fig)
    del values
    gc.collect()

plot_raw("TS1", "cooler", "03_cooler_TS1_signals")
plot_raw("CE", "cooler", "03_cooler_CE_signals")
plot_raw("PS2", "valve", "04_valve_PS2_signals")
plot_raw("EPS1", "pump", "05_pump_EPS1_signals")
plot_raw("FS1", "pump", "05_pump_FS1_signals")
plot_raw("PS1", "accumulator", "06_accumulator_PS1_signals")

### Observation
The exact example row indices are printed above. Cooler examples use cooler values 3, 20, and 100 while trying to hold valve=100, pump=0, and accumulator=130; valve, pump, and accumulator examples use the same healthy control strategy. Closest available cycles were used where the exact combination was absent.

## 5. Class-average signal overlays

In [7]:
def plot_class_average(sensor, target, filename):
    values = load_sensor(sensor)
    meta = config.SENSORS[sensor]
    time = pd.Series(range(values.shape[1])) / meta["sampling_hz"]
    fig, ax = plt.subplots(figsize=(11, 4))
    for value in sorted(profile[target].unique()):
        mask = profile[target].eq(value).to_numpy()
        ax.plot(time, values[mask].mean(axis=0), label=f"{target}={value}")
    ax.set(xlabel="seconds", ylabel=meta["unit"], title=f"Average {sensor} by {target}")
    ax.legend(ncol=2)
    save_figure(fig, filename)
    plt.close(fig)
    del values
    gc.collect()

plot_class_average("CE", "cooler", "07_mean_overlay_cooler_CE")
plot_class_average("PS2", "valve", "07_mean_overlay_valve_PS2")
plot_class_average("EPS1", "pump", "07_mean_overlay_pump_EPS1")
plot_class_average("PS1", "accumulator", "07_mean_overlay_accumulator_PS1")

### Observation
The class-average overlays summarize all cycles for one informative sensor per component. The plots are descriptive and do not establish a chronological failure progression.

## 6. Sensor mean correlations

In [8]:
correlation = means.corr(method="pearson")
fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(correlation, cmap="vlag", center=0, ax=ax)
save_figure(fig, "08_sensor_mean_correlation_heatmap")
plt.close(fig)

pairs = correlation.where(~pd.DataFrame(False, index=correlation.index, columns=correlation.columns)).stack().rename("pearson_r").reset_index()
pairs.columns = ["sensor_1", "sensor_2", "pearson_r"]
strong = pairs[(pairs.sensor_1 < pairs.sensor_2) & (pairs.pearson_r.abs() > 0.9)].sort_values("pearson_r", key=lambda s: s.abs(), ascending=False)
strong.to_csv(config.RESULTS_DIR / "strong_correlations.csv", index=False)
display(strong)

,sensor_1,sensor_2,pearson_r
73,PS5,PS6,0.999993
164,TS1,TS3,0.999732
163,TS1,TS2,0.999344
165,TS1,TS4,0.999242
181,TS2,TS3,0.999067
199,TS3,TS4,0.998395
182,TS2,TS4,0.998292
147,FS2,TS3,-0.996281
1,PS1,PS2,0.995294
80,PS5,TS4,-0.995100


### Observation
PS5 and PS6 have Pearson r = 0.999993 for per-cycle means. The temperature sensors TS1-TS4 are also strongly correlated, and FS2 is strongly negative with temperature and several pressure sensors. These are correlations, not duplicate-feature removal decisions.

## 7. Sensor-versus-target relationships

In [9]:
relevant = {"cooler": ["CE", "CP", "TS1"], "valve": ["PS1", "PS2", "FS1"],
            "pump": ["EPS1", "FS1", "FS2"], "accumulator": ["PS1", "PS2", "PS4"]}
group_outputs = []
for target, sensors in relevant.items():
    data = profile[[target]].join(means[sensors])
    group_means = data.groupby(target)[sensors].mean()
    group_means.to_csv(config.RESULTS_DIR / f"group_means_{target}.csv")
    group_outputs.append(group_means.assign(target=target).reset_index())
    fig, axes = plt.subplots(1, len(sensors), figsize=(5 * len(sensors), 4))
    for ax, sensor in zip(axes.ravel(), sensors):
        sns.boxplot(data=data, x=target, y=sensor, ax=ax)
    save_figure(fig, f"09_relationships_{target}")
    plt.close(fig)
    display(pd.DataFrame({"target": [target] * len(group_means), "class": group_means.index}).join(group_means.reset_index(drop=True)))

pd.concat(group_outputs, ignore_index=True).to_csv(config.RESULTS_DIR / "target_group_means.csv", index=False)

spearman = means.join(profile[config.TARGET_COLUMNS[:4]]).corr(method="spearman").loc[
    means.columns, config.TARGET_COLUMNS[:4]
]
spearman_long = spearman.stack().rename("spearman_r").reset_index()
spearman_long.columns = ["sensor", "target", "spearman_r"]
spearman_long["abs_r"] = spearman_long["spearman_r"].abs()
spearman_long = spearman_long.sort_values("abs_r", ascending=False)
spearman_long.to_csv(config.RESULTS_DIR / "sensor_target_spearman.csv", index=False)
display(spearman_long.head(20))

,target,class,CE,CP,TS1
0,cooler,3,19.572635,1.511871,54.960531
1,cooler,20,27.201670,1.743498,45.474273
2,cooler,100,46.930734,2.165437,35.955322


,target,class,PS1,PS2,FS1
0,valve,73,161.408184,109.810641,5.951552
1,valve,80,161.211482,109.957396,5.993925
2,valve,90,161.239718,110.273763,5.990347
3,valve,100,159.716214,108.771240,6.409693


,target,class,EPS1,FS1,FS2
0,pump,0,2467.495952,6.602685,9.667347
1,pump,1,2519.311526,5.787250,9.618077
2,pump,2,2541.227665,5.606904,9.636420


,target,class,PS1,PS2,PS4
0,accumulator,90,160.633679,109.470717,2.222887
1,accumulator,100,162.393704,111.338733,3.364234
2,accumulator,115,161.525905,110.422057,3.136693
3,accumulator,130,158.320840,107.258429,2.243112


,sensor,target,spearman_r,abs_r
56,CE,cooler,0.942164,0.942164
60,CP,cooler,0.941729,0.941729
48,TS4,cooler,-0.935601,0.935601
20,PS6,cooler,0.934964,0.934964
16,PS5,cooler,0.934927,0.934927
36,TS1,cooler,-0.932126,0.932126
40,TS2,cooler,-0.929701,0.929701
44,TS3,cooler,-0.929167,0.929167
52,VS1,cooler,-0.923480,0.923480
32,FS2,cooler,0.916623,0.916623


### Observation
CE and CP have Spearman relationships of about 0.942 with cooler condition. SE and FS1 are strongly related to pump condition. Accumulator relationships are weaker among the selected pressure sensors, with PS2 at r_s = -0.302 and PS1 at -0.283.

## 8. Stable flag comparison

In [10]:
stable_data = profile[["stable"]].join(means[["CE", "PS2", "EPS1", "TS1"]])
stable_comparison = stable_data.groupby("stable")[["CE", "PS2", "EPS1", "TS1"]].agg(["count", "mean", "std"])
stable_comparison.to_csv(config.RESULTS_DIR / "stable_flag_comparison.csv")
display(stable_comparison)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, sensor in zip(axes, ["CE", "PS2", "EPS1", "TS1"]):
    sns.boxplot(data=stable_data, x="stable", y=sensor, ax=ax)
save_figure(fig, "10_stable_flag_comparison")
plt.close(fig)

CE                         PS2                        EPS1  \
       count       mean        std count        mean       std count   
stable                                                                 
0       1449  31.427104  11.606659  1449  110.124318  5.845358  1449   
1        756  31.053693  11.518727   756  107.953118  1.989113   756   

                                 TS1                       
               mean        std count       mean       std  
stable                                                     
0       2509.780593  76.145072  1449  45.592059  8.311159  
1       2468.155767  60.456465   756  45.103539  7.336022

### Observation
For stable=0 versus stable=1, CE means are 31.427 versus 31.054, PS2 means 110.124 versus 107.953, EPS1 means 2509.781 versus 2468.156, and TS1 means 45.592 versus 45.104. The flag may need review in later modeling because it is a possible measurement-state indicator, but no conclusion is drawn here.

## 9. Report-only IQR outliers

In [11]:
mean_columns = [column for column in summary.columns if column.endswith("_mean")]
max_columns = [column for column in summary.columns if column.endswith("_max") and column.split("_", 1)[0] in ([f"PS{i}" for i in range(1, 7)] + ["EPS1"])]
outliers = count_outliers_iqr(summary[mean_columns + max_columns])
outliers.to_csv(config.RESULTS_DIR / "outlier_summary_iqr.csv", index=False)
display(outliers)

,column,q1,q3,iqr,lower_fence,upper_fence,outlier_count
0,PS1_mean,158.100195,161.000736,2.900541,153.749384,165.351547,240
1,PS2_mean,106.962383,109.421612,2.459230,103.273538,113.110457,240
2,PS3_mean,1.729733,1.932047,0.202314,1.426263,2.235517,240
3,PS4_mean,0.000000,3.503266,3.503266,-5.254899,8.758165,532
4,PS5_mean,8.547239,9.844351,1.297113,6.601570,11.790020,0
5,PS6_mean,8.487167,9.729276,1.242109,6.624004,11.592439,0
6,EPS1_mean,2442.933474,2548.211492,105.278018,2285.016447,2706.128520,10
7,FS1_mean,6.391670,6.657508,0.265838,5.992913,7.056266,240
8,FS2_mean,9.203397,10.155008,0.951612,7.775979,11.582426,0
9,TS1_mean,36.237150,54.104316,17.867167,9.436400,80.905066,0


### Observation
The IQR rule identifies 240 outliers in per-cycle mean PS1, PS2, PS3, FS1, and SE, 532 in PS4 mean, 10 in EPS1 mean, and 5 in VS1 mean. PS2 maximum also has 46. These extreme values may represent real operating conditions and are not removed or modified.

## 10. Findings
- Cooler, valve, pump, and accumulator labels are grouped by row rather than randomly ordered.
- The valve label is the most imbalanced among the four component targets (largest/smallest = 3.125).
- The 17 sensor files have 2205 aligned cycles and no NaN or inf values.
- PS4 has 1237 duplicate cycles and 1238 constant cycles; SE has 8 constant columns.
- Several sensor means are highly correlated, especially PS5/PS6 and TS1-TS4.
- Cooler-related sensors have strong Spearman relationships, while accumulator relationships are weaker.
- Extreme IQR counts are retained as report-only findings.
- Condition labels do not imply chronological failure progression; they describe controlled test-rig states on a single experimental test rig.